# Paired Bootstrap & Threshold-Transfer Analysis

Reproduces shared 90%-specificity threshold calibration on PROVe-AI calibration nevi, cross-cohort evaluation, and patient-clustered paired bootstrap confidence intervals (`draws=2000`, `seed=23`) from saved prediction CSVs—no GPU required.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from final_benchmark_intervals import paired_results
from final_external_benchmark import PRIMARY, shared_cutoffs, summarize_all_datasets

In [ ]:
# Point PRED_DIR to your directory containing isic_predictions.csv, derm_predictions.csv,
# prove_predictions.csv, and ph2_predictions.csv.
PRED_DIR = ROOT / "outputs"

if (PRED_DIR / "prove_predictions.csv").exists():
    isic_df = pd.read_csv(PRED_DIR / "isic_predictions.csv")
    derm_df = pd.read_csv(PRED_DIR / "derm_predictions.csv")
    prove_df = pd.read_csv(PRED_DIR / "prove_predictions.csv")
    ph2_df = pd.read_csv(PRED_DIR / "ph2_predictions.csv")

    for df in (isic_df, derm_df, prove_df, ph2_df):
        if "blend_60_40" not in df.columns and {"retained", "balanced_mel_nv"}.issubset(df.columns):
            df["blend_60_40"] = 0.60 * df["retained"] + 0.40 * df["balanced_mel_nv"]

    cal = prove_df.loc[prove_df.split.eq("calibration") & prove_df.cohort.eq("nevus")].copy()
    cutoffs = shared_cutoffs(cal, [m for m in (*PRIMARY, "committee29", "adae") if m in cal.columns])
    mn_df, mimic_df = summarize_all_datasets(cutoffs, isic_df, derm_df, prove_df, ph2_df)
    display(mn_df.round(4))
    display(mimic_df.round(3))

    test_prove = prove_df.loc[prove_df.split.eq("test") & prove_df.cohort.isin(["melanoma", "nevus"])]
    paired_prove = paired_results(test_prove, test_prove.cohort.eq("melanoma"), cutoffs, "PROVe-AI")
    display(paired_prove.round(4))
else:
    print(f"Place prediction CSVs in {PRED_DIR} or run scripts/run_paired_analysis.py.")